# Notebook Thử Nghiệm: Khảo Sát & Ghép Bảng Dữ Liệu Bronze (Listings + Details)

Notebook này dùng để thực hiện **từng bước khảo sát dữ liệu thực tế** trước khi làm sạch và chuẩn hóa sang tầng Silver:

1.  **Nạp Dữ Liệu Master Bronze**: Đọc 2 bảng `bronze_listings_master.json` và `bronze_details_valid.json`.
2.  **Ghép Bảng (Join Listings + Details)**: Kết hợp các trường từ trang danh mục (Logo, tags ngoài) và trang chi tiết (JD sâu, chế độ, yêu cầu).
3.  **Thống Kê Chi Tiết**: Phân tích cấu trúc (`info`), giá trị rỗng/khuyết (`null`/chuỗi rỗng), và trùng lặp (`duplicates`).
4.  **Xem Mẫu Dữ Liệu (Sample 10 Dòng)**: Khảo sát trực quan 10 bản ghi sau khi ghép.

In [2]:
import os
import json
import pandas as pd
import numpy as np
from IPython.display import display

# Cấu hình hiển thị pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)
print(' Môi trường phân tích dữ liệu đã sẵn sàng!')

 Môi trường phân tích dữ liệu đã sẵn sàng!


## 1. Nạp Dữ Liệu Master Từ Tầng Bronze

Đọc 2 tệp tích lũy chuẩn từ `data/bronze/`:
- `bronze_listings_master.json`: Bảng danh mục tin tuyển dụng (kèm tags, logo, trạng thái crawl).
- `bronze_details_valid.json`: Bảng chi tiết việc làm hợp lệ sau khi qua bộ kiểm định Data Quality.

In [4]:
# Đường dẫn tới 2 file Bronze Master
listings_path = os.path.abspath('../data/bronze/bronze_listings_master.json')
if not os.path.exists(listings_path):
    listings_path = os.path.abspath('data/bronze/bronze_listings_master.json')

details_path = os.path.abspath('../data/bronze/bronze_details_valid.json')
if not os.path.exists(details_path):
    details_path = os.path.abspath('data/bronze/bronze_details_valid.json')

with open(listings_path, 'r', encoding='utf-8') as f:
    listings_data = json.load(f)
with open(details_path, 'r', encoding='utf-8') as f:
    details_data = json.load(f)

df_listings = pd.DataFrame(listings_data)
df_details = pd.DataFrame(details_data)

print(f' Đang đọc Listings từ: {listings_path}')
print(f' Đang đọc Details từ : {details_path}\n')
print(f' Bảng Listings: {len(df_listings):,} dòng, {df_listings.shape[1]} cột')
print(f' Bảng Details : {len(df_details):,} dòng, {df_details.shape[1]} cột\n')

# Thống kê số lượng tin theo từng sàn
summary_sources = pd.DataFrame({
    'Số Listings': df_listings['source'].value_counts(),
    'Số Details Valid': df_details['source'].value_counts()
})
display(summary_sources)

 Đang đọc Listings từ: C:\Users\Minh Doan\repo_github\IT-Job-Market-Intelligence\data\bronze\bronze_listings_master.json
 Đang đọc Details từ : C:\Users\Minh Doan\repo_github\IT-Job-Market-Intelligence\data\bronze\bronze_details_valid.json

 Bảng Listings: 2,327 dòng, 20 cột
 Bảng Details : 2,327 dòng, 35 cột

              Số Listings  Số Details Valid
source                                     
topcv                1161              1161
vietnamworks         1046              1046
topdev                120               120


## 2. Ghép 2 Bảng (Join Listings + Details)

Ghép theo cặp khóa định danh duy nhất: `['source', 'source_job_id']`.
- **Mục tiêu**: Bổ sung các cột giá trị từ Listings mà Details có thể chưa có (như `company_logo_url`, `skills_raw`, `posted_at_raw`, `raw_file_name` của listing).
- Giữ nguyên toàn bộ các trường nội dung chi tiết sâu (`overview_raw`, `responsibilities_raw`, `requirements_raw`, `benefits_raw`, `work_mode_raw`, `company_size_raw`...) từ Details.

In [6]:
# Chọn các cột bổ sung từ Listings (tránh xung đột trùng lặp cột)
listing_extra_cols = ['source', 'source_job_id', 'company_logo_url', 'skills_raw', 'posted_at_raw', 'raw_file_name']
df_list_sub = df_listings[listing_extra_cols].drop_duplicates(subset=['source', 'source_job_id']).rename(
    columns={'raw_file_name': 'listing_file_name'}
)

# Ghép bảng Details với Listings theo ['source', 'source_job_id']
df_merged = pd.merge(
    df_details,
    df_list_sub,
    on=['source', 'source_job_id'],
    how='inner'
)

print(f' Kết quả ghép bảng (Join):')
print(f'  • Tổng số dòng sau khi ghép : {len(df_merged):,} dòng')
print(f'  • Tổng số cột sau khi ghép  : {df_merged.shape[1]} cột')
print(f'  • Tỷ lệ khớp khóa (Match)  : {len(df_merged)/len(df_details)*100:.2f}% (Khớp hoàn toàn 100%)')

 Kết quả ghép bảng (Join):
  • Tổng số dòng sau khi ghép : 2,327 dòng
  • Tổng số cột sau khi ghép  : 39 cột
  • Tỷ lệ khớp khóa (Match)  : 100.00% (Khớp hoàn toàn 100%)


## 3. Thống Kê Chi Tiết: Info, Missing Values (Null / Rỗng), và Duplicates (Dup)

Khảo sát toàn diện chất lượng dữ liệu của bảng sau khi ghép:
1. **Cấu trúc & Kiểu dữ liệu (`.info()`)**.
2. **Thống kê giá trị khuyết (`null` và chuỗi rỗng `""`)**: Tính số lượng và tỷ lệ % khuyết trên từng trường dữ liệu.
3. **Kiểm tra trùng lặp (`duplicates`)**: Kiểm tra trùng theo ID nguồn, theo URL, và theo cặp chức danh - công ty.

In [8]:
# 1. Hiển thị thông tin tổng quan kiểu dữ liệu
print('--- 1. THÔNG TIN CẤU TRÚC (DF INFO) ---')
df_merged.info(verbose=True, show_counts=True)

# 2. Thống kê Missing Values (Null / None và Chuỗi Rỗng "")
null_counts = df_merged.isnull().sum()
empty_counts = (df_merged == '').sum()
total_missing = null_counts + empty_counts
missing_pct = (total_missing / len(df_merged) * 100).round(2)

df_missing_report = pd.DataFrame({
    'Kiểu Dữ Liệu': df_merged.dtypes.astype(str),
    'Số lượng Null/None': null_counts,
    'Số lượng Chuỗi Rỗng ("")': empty_counts,
    'Tổng Số Khuyết': total_missing,
    'Tỷ Lệ Khuyết (%)': missing_pct
}).sort_values(by='Tỷ Lệ Khuyết (%)', ascending=False)

print('\n--- 2. BẢNG THỐNG KÊ GIÁ TRỊ KHUYẾT (MISSING / NULL REPORT) ---')
display(df_missing_report)

# 3. Kiểm tra Trùng lặp (Duplicates Report)
dup_source_id = df_merged.duplicated(subset=['source', 'source_job_id']).sum()
dup_job_url = df_merged.duplicated(subset=['job_url']).sum()
dup_title_company = df_merged.duplicated(subset=['title_raw', 'company_name_raw']).sum()

print('--- 3. BẢNG KIỂM TRA TRÙNG LẶP (DUPLICATES REPORT) ---')
df_dup_report = pd.DataFrame({
    'Tiêu Chí Trùng Lặp': [
        'Theo cặp khóa nguồn [source, source_job_id]',
        'Theo đường dẫn [job_url]',
        'Theo cùng vị trí & cùng công ty [title_raw, company_name_raw]'
    ],
    'Số Bản Ghi Trùng': [dup_source_id, dup_job_url, dup_title_company],
    'Tỷ Lệ Trùng (%)': [
        round(dup_source_id / len(df_merged) * 100, 2),
        round(dup_job_url / len(df_merged) * 100, 2),
        round(dup_title_company / len(df_merged) * 100, 2)
    ]
})
display(df_dup_report)

--- 1. THÔNG TIN CẤU TRÚC (DF INFO) ---
<class 'pandas.DataFrame'>
RangeIndex: 2327 entries, 0 to 2326
Data columns (total 39 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   detail_id             2327 non-null   str   
 1   source                2327 non-null   str   
 2   source_job_id         2327 non-null   str   
 3   job_url               2327 non-null   str   
 4   title_raw             2327 non-null   str   
 5   company_name_raw      2327 non-null   str   
 6   company_url           2327 non-null   str   
 7   company_size_raw      2327 non-null   str   
 8   company_location_raw  2327 non-null   str   
 9   industry_raw          2327 non-null   str   
 10  salary_raw            2327 non-null   str   
 11  experience_raw        2327 non-null   str   
 12  education_raw         2327 non-null   str   
 13  location_raw          2327 non-null   str   
 14  work_mode_raw         2327 non-null   str   
 15  employmen

## 4. Xem Mẫu Dữ Liệu Sau Khi Ghép (Toàn Bộ 39 Cột - Dễ Quan Sát)

Vì dữ liệu sau khi ghép có **39 cột** (bao gồm cả các trường text dài và metadata), nếu để một bảng phẳng nằm ngang sẽ rất khó quan sát. Chúng ta tổ chức hiển thị theo các cách sau để bạn dễ kiểm tra nhất:

1.  **Cách 1: Hiển thị Bảng Chuyển Vị (Transposed `.T`)**: Chuyển toàn bộ 39 cột thành danh sách dọc. Mỗi tin là một cột  Dễ dàng cuộn dọc đọc trọn vẹn từng trường mà không bị tràn màn hình.
2.  **Cách 2: Hiển thị 10 Dòng Theo Từng Nhóm Nghiệp Vụ**: Phân tách 39 cột thành 5 nhóm có cấu trúc (*Định danh, Công ty, Lương & Đãi ngộ, Kỹ năng, Nội dung JD, Data Quality*).
3.  **Cách 3: Bảng Phẳng Đầy Đủ (Toàn bộ 39 cột x 10 dòng)**: Mở rộng `display.max_columns = None` để bạn có thể cuộn ngang xem tất cả các cột nếu cần.

In [10]:
# --- CÁCH 1: HIỂN THỊ DẠNG CHUYỂN VỊ (TRANSPOSED VIEW) ---
# 39 cột sẽ hiển thị theo chiều dọc (Index), các mẫu tin hiển thị theo cột
print(' [CÁCH 1] XEM DỌC TOÀN BỘ 39 CỘT CỦA 3 TIN TUYỂN DỤNG MẪU:')
sample_3 = df_merged.sample(3, random_state=42)
display(sample_3.T)


 [CÁCH 1] XEM DỌC TOÀN BỘ 39 CỘT CỦA 3 TIN TUYỂN DỤNG MẪU:
                                                                             1736  \
detail_id                                                           topcv_2318361   
source                                                                      topcv   
source_job_id                                                             2318361   
job_url               https://www.topcv.vn/viec-lam/graphic-designer-intern/23...   
title_raw                                                 Graphic Designer Intern   
company_name_raw                           Công ty TNHH Đầu tư và Thương Mại Elva   
company_url           https://www.topcv.vn/cong-ty/cong-ty-tnhh-dau-tu-va-thuo...   
company_size_raw                                                  25-99 nhân viên   
company_location_raw  Elva Tower, 211 Trần Não, Phường An Khánh, Tp. Thủ Đức, ...   
industry_raw          Thiết kế đồ họa, Marketing / Truyền thông / Quảng cáo, I...   
salary

In [11]:
# --- CÁCH 2: XEM MẪU 10 DÒNG THEO TỪNG NHÓM NGHIỆP VỤ ---
sample_10 = df_merged.sample(10, random_state=42)

# Nhóm 1: Định danh & Truy vết nguồn (7 cột)
group_identity = ['source', 'source_job_id', 'job_url', 'crawled_at', 'raw_date', 'raw_file_name', 'listing_file_name']
print(' NHÓM 1: ĐỊNH DANH & TRUY VẾT NGUỒN (7 CỘT)')
display(sample_10[group_identity])

# Nhóm 2: Thông tin Công ty (7 cột)
group_company = ['company_name_raw', 'company_url', 'company_logo_url', 'company_size_raw', 'company_location_raw', 'industry_raw', 'location_raw']
print('\n NHÓM 2: THÔNG TIN DOANH NGHIỆP (7 CỘT)')
display(sample_10[group_company])

# Nhóm 3: Vị trí, Lương & Đãi ngộ tuyển dụng (8 cột)
group_job_specs = ['title_raw', 'salary_raw', 'experience_raw', 'education_raw', 'work_mode_raw', 'employment_type_raw', 'working_time_raw', 'deadline_raw']
print('\n NHÓM 3: VỊ TRÍ, LƯƠNG & ĐIỀU KIỆN TUYỂN DỤNG (8 CỘT)')
display(sample_10[group_job_specs])

# Nhóm 4: Kỹ năng & Ngày tháng (6 cột)
group_skills = ['skills_raw', 'required_skills_raw', 'preferred_skills_raw', 'posted_at_raw', 'published_at_raw', 'updated_at_raw']
print('\n NHÓM 4: KỸ NĂNG & MỐC THỜI GIAN (6 CỘT)')
display(sample_10[group_skills])

# Nhóm 5: Nội dung chi tiết văn bản (4 cột)
group_texts = ['overview_raw', 'responsibilities_raw', 'requirements_raw', 'benefits_raw']
print('\n NHÓM 5: NỘI DUNG VĂN BẢN CHI TIẾT (4 CỘT - TRÍCH ĐOẠN 80 KÝ TỰ)')
display(sample_10[['title_raw'] + group_texts].map(lambda x: (str(x)[:80] + '...') if len(str(x)) > 80 else str(x)))


 NHÓM 1: ĐỊNH DANH & TRUY VẾT NGUỒN (7 CỘT)
            source source_job_id  \
1736         topcv       2318361   
755         topdev       2134741   
1669  vietnamworks       2113740   
1322  vietnamworks       2116086   
1870  vietnamworks       2090843   
1004  vietnamworks       2109326   
1900         topcv       2328694   
1501  vietnamworks       2116664   
1222  vietnamworks       2117699   
1173  vietnamworks       2098089   

                                                          job_url  \
1736  https://www.topcv.vn/viec-lam/graphic-designer-intern/23...   
755   https://topdev.vn/viec-lam/web-developer-backend-fullsta...   
1669  https://www.vietnamworks.com/ky-su-bao-mat-he-thong-syst...   
1322  https://www.vietnamworks.com/industry-manager-gaming-lar...   
1870  https://www.vietnamworks.com/ky-su-kiem-soat-chat-luong-...   
1004  https://www.vietnamworks.com/chuyen-gia-quan-ly-hieu-qua...   
1900  https://www.topcv.vn/viec-lam/it-service-manager/2328694...   
1501  h

In [12]:
# --- CÁCH 3: BẢNG PHẲNG ĐẦY ĐỦ TOÀN BỘ 39 CỘT (SAMPLE 10 DÒNG) ---
print(' [CÁCH 3] BẢNG PHẲNG ĐẦY ĐỦ TẤT CẢ 39 CỘT (CÓ THỂ CUỘN NGANG ĐỂ XEM TOÀN BỘ):')
display(sample_10)


 [CÁCH 3] BẢNG PHẲNG ĐẦY ĐỦ TẤT CẢ 39 CỘT (CÓ THỂ CUỘN NGANG ĐỂ XEM TOÀN BỘ):
                 detail_id        source source_job_id  \
1736         topcv_2318361         topcv       2318361   
755         topdev_2134741        topdev       2134741   
1669  vietnamworks_2113740  vietnamworks       2113740   
1322  vietnamworks_2116086  vietnamworks       2116086   
1870  vietnamworks_2090843  vietnamworks       2090843   
1004  vietnamworks_2109326  vietnamworks       2109326   
1900         topcv_2328694         topcv       2328694   
1501  vietnamworks_2116664  vietnamworks       2116664   
1222  vietnamworks_2117699  vietnamworks       2117699   
1173  vietnamworks_2098089  vietnamworks       2098089   

                                                          job_url  \
1736  https://www.topcv.vn/viec-lam/graphic-designer-intern/23...   
755   https://topdev.vn/viec-lam/web-developer-backend-fullsta...   
1669  https://www.vietnamworks.com/ky-su-bao-mat-he-thong-syst...   
1322  h